# Original Data Exact Formula
This notebook discovers the original data's exact formula. This month's Kaggle playground competition is synthetic data generated from an "original data" [here][1] (And I made a copy [here][2] in case it gets deleted). This original data has 19 columns but @aerdem4 showed [here][3] that only 6 are meaningful. Through careful EDA analysis, I determined the original data's formula and demonstrate it below.

Discussion about this notebook is [here][4]

[1]: https://www.kaggle.com/datasets/miadul/irrigation-water-requirement-prediction-dataset/data
[2]: https://www.kaggle.com/datasets/cdeotte/s6e4-the-original-dataset
[3]: https://www.kaggle.com/competitions/playground-series-s6e4/discussion/687104
[4]: https://www.kaggle.com/competitions/playground-series-s6e4/discussion/687460

In [ ]:
import pandas as pd
import numpy as np
import warnings
import gc

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, recall_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.linear_model import LogisticRegression

# Suppress warnings
warnings.filterwarnings("ignore")

# ---------------------------------------------------------
# 1. LOAD DATA & SETUP
# ---------------------------------------------------------
print("Loading data...")

train = pd.read_csv("/kaggle/input/datasets/cdeotte/s6e4-the-original-dataset/irrigation_prediction.csv")

TARGET = "Irrigation_Need"
target_mapping = {"Low": 0, "Medium": 1, "High": 2}
inverse_target_mapping = {0: "Low", 1: "Medium", 2: "High"}

train[TARGET] = train[TARGET].map(target_mapping)

# ---------------------------------------------------------
# 2. FEATURE ENGINEERING
# ---------------------------------------------------------
print("Engineering features...")

def create_features(df):
    df = df.copy()

    # 4 boolean numeric features based on threshold insights
    df["soil_lt_25"] = (df["Soil_Moisture"] < 25).astype(int)
    df["temp_gt_30"] = (df["Temperature_C"] > 30).astype(int)
    df["rain_lt_300"] = (df["Rainfall_mm"] < 300).astype(int)
    df["wind_gt_10"] = (df["Wind_Speed_kmh"] > 10).astype(int)

    return df

train = create_features(train)

NUM_FEATURES = [
    "soil_lt_25",
    "temp_gt_30",
    "rain_lt_300",
    "wind_gt_10",
]

CAT_FEATURES = [
    "Crop_Growth_Stage",
    "Mulching_Used",
]

# ---------------------------------------------------------
# 3. ONE-HOT ENCODE FEATURES
# ---------------------------------------------------------
print("Creating one-hot encoded features...")

X = pd.get_dummies(
    train[NUM_FEATURES + CAT_FEATURES],
    columns=CAT_FEATURES,
    drop_first=False
)

FEATURES = X.columns.tolist()
y = train[TARGET].copy()

print(f"Total engineered features: {len(FEATURES)}")
print("Features:")
for f in FEATURES:
    print(f"  {f}")

# ---------------------------------------------------------
# 4. TRAINING WITH 5-FOLD CV
# ---------------------------------------------------------
print("\nTraining Logistic Regression with 5-Fold CV...")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

logreg_oof = np.zeros((len(train), 3))
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
    print(f"\n{'=' * 40}\nFold {fold}/5\n{'=' * 40}")

    X_tr = X.iloc[train_idx].copy()
    y_tr = y.iloc[train_idx].copy()

    X_val = X.iloc[val_idx].copy()
    y_val = y.iloc[val_idx].copy()

    # Class-balanced sample weights
    classes = np.unique(y_tr)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_tr)
    class_weight_dict = dict(zip(classes, weights))
    sample_weights = np.array([class_weight_dict[label] for label in y_tr])

    # Model
    model = LogisticRegression(
        multi_class="multinomial",
        solver="lbfgs",
        max_iter=1000,
        random_state=42,
        n_jobs=-1
    )

    # Train
    model.fit(X_tr, y_tr, sample_weight=sample_weights)

    # Predict
    val_pred_proba = model.predict_proba(X_val)
    logreg_oof[val_idx] = val_pred_proba

    # Fold metric
    val_pred = np.argmax(val_pred_proba, axis=1)
    fold_acc = balanced_accuracy_score(y_val, val_pred)
    fold_scores.append(fold_acc)
    print(f"Fold {fold} Balanced Accuracy: {fold_acc:.5f}")

    gc.collect()

# ---------------------------------------------------------
# 5. OVERALL METRICS
# ---------------------------------------------------------
oof_pred = np.argmax(logreg_oof, axis=1)

overall_acc = balanced_accuracy_score(train[TARGET], oof_pred)
print(f"\nOverall CV Balanced Accuracy: {overall_acc:.5f}")
print(f"Mean Fold Balanced Accuracy : {np.mean(fold_scores):.5f}")
print(f"Std Fold Balanced Accuracy  : {np.std(fold_scores):.5f}")

# Recall for each class
class_recalls = recall_score(
    train[TARGET],
    oof_pred,
    labels=[0, 1, 2],
    average=None
)

print("\nRecall by target class:")
for class_id, recall_val in zip([0, 1, 2], class_recalls):
    print(f"  {inverse_target_mapping[class_id]}: {recall_val:.5f}")

# Display Formula

In [ ]:
classes = np.unique(y)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y)
class_weight_dict = dict(zip(classes, weights))
sample_weights = np.array([class_weight_dict[label] for label in y])

_ = model.fit(X, y, sample_weight=sample_weights)

print("\nLogistic Regression Formula:\n")

feature_names = FEATURES  # from your pipeline

coefs = model.coef_        # shape: (n_classes, n_features)
intercepts = model.intercept_

for class_idx in range(coefs.shape[0]):
    class_name = inverse_target_mapping[class_idx]

    print(f"\nClass: {class_name}")
    print("-" * 50)

    terms = []
    for coef, fname in zip(coefs[class_idx], feature_names):
        terms.append(f"({coef:.4f} * {fname})")

    equation = " + ".join(terms)

    print(f"logit(P(y={class_name})) = {intercepts[class_idx]:.4f} + {equation}")

# EDA of Features

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# 1. LOAD DATA
# ---------------------------------------------------------
train = pd.read_csv("/kaggle/input/datasets/cdeotte/s6e4-the-original-dataset/irrigation_prediction.csv")

TARGET = "Irrigation_Need"

target_mapping = {"Low": 0, "Medium": 1, "High": 2}
train["target_num"] = train[TARGET].map(target_mapping)

# ---------------------------------------------------------
# 2. DEFINE FEATURES + THRESHOLDS
# ---------------------------------------------------------
feature_config = {
    "Soil_Moisture": 25,
    "Temperature_C": 30,
    "Rainfall_mm": 300,
    "Wind_Speed_kmh": 10,
}

n_bins = 100

# ---------------------------------------------------------
# 3. LOOP THROUGH FEATURES
# ---------------------------------------------------------
for col, threshold in feature_config.items():

    df = train.copy()

    # Quantile bins (more stable)
    df["bin"] = pd.qcut(df[col], q=n_bins, duplicates="drop")

    # Aggregate stats
    bin_stats = (
        df.groupby("bin", observed=False)
        .agg(
            count=(col, "size"),
            mean_target=("target_num", "mean"),
            bin_left=(col, "min"),
            bin_right=(col, "max"),
        )
        .reset_index()
    )

    bin_stats["bin_center"] = (bin_stats["bin_left"] + bin_stats["bin_right"]) / 2

    # ---------------------------------------------------------
    # PLOT
    # ---------------------------------------------------------
    fig, ax1 = plt.subplots(figsize=(14, 6))

    # Bar plot (counts)
    ax1.bar(
        bin_stats["bin_center"],
        bin_stats["count"],
        width=(bin_stats["bin_right"] - bin_stats["bin_left"]).values,
        alpha=0.6,
        align="center"
    )
    ax1.set_xlabel(col)
    ax1.set_ylabel("Count")
    ax1.set_title(f"{col}: {n_bins} bins vs Mean Target")

    # Mean target line
    ax2 = ax1.twinx()
    ax2.plot(
        bin_stats["bin_center"],
        bin_stats["mean_target"],
        marker="o",
        linewidth=2
    )
    ax2.set_ylabel("Mean target")
    ax2.set_ylim(-0.05, 2.05)

    # ---------------------------------------------------------
    # THRESHOLD LINE
    # ---------------------------------------------------------
    ax1.axvline(
        x=threshold,
        linestyle="--",
        linewidth=2,
    )

    # Annotate threshold
    ax1.text(
        threshold,
        ax1.get_ylim()[1] * 0.9,
        f"Threshold = {threshold}",
        rotation=90,
        verticalalignment="top"
    )

    plt.tight_layout()
    plt.show()